## Setup and Imports

In [65]:
# Standard library imports
import sys
from pathlib import Path
from datetime import datetime

# Third-party imports
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project imports
from ayne.utils.query_utils import (
    get_database_summary_stats,
    get_enrichment_status_by_year,
    get_movies_due_for_update_by_year,
    get_data_quality_metrics,
    get_recent_update_activity,
    execute_custom_query
)

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

print(f"✓ Setup complete - {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

✓ Setup complete - 2025-12-01 23:44:55


## 1. Database Overview

High-level statistics about the entire database.

In [66]:
# Get summary statistics
stats = get_database_summary_stats()

# Display as formatted output
print("="*60)
print(" " * 15 + "DATABASE SUMMARY STATISTICS")
print("="*60)
print(f"\n📊 Total Movies: {stats['total_movies']:,}")
print(f"   └─ Date Range: {stats['earliest_movie']} to {stats['latest_movie']}")
print(f"   └─ Unique Years: {stats['unique_years']}")
print(f"\n🔑 ID Coverage:")
print(f"   └─ With TMDB ID: {stats['with_tmdb_id']:,} ({100.0 * stats['with_tmdb_id'] / stats['total_movies']:.1f}%)")
print(f"   └─ With IMDB ID: {stats['with_imdb_id']:,} ({100.0 * stats['with_imdb_id'] / stats['total_movies']:.1f}%)")
print(f"\n✨ Enrichment Status:")
print(f"   └─ TMDB Enriched: {stats['enriched_tmdb']:,} ({stats['tmdb_enrichment_pct']:.1f}%)")
print(f"   └─ OMDB Enriched: {stats['enriched_omdb']:,} ({stats['omdb_enrichment_pct']:.1f}%)")
print(f"   └─ Fully Refreshed: {stats['fully_refreshed']:,} ({100.0 * stats['fully_refreshed'] / stats['total_movies']:.1f}%)")
print(f"\n❄️  Frozen Movies: {stats['frozen_movies']:,}")
print("="*60)

               DATABASE SUMMARY STATISTICS

📊 Total Movies: 32,534
   └─ Date Range: 1950-01-01 00:00:00 to 2025-11-16 00:00:00
   └─ Unique Years: 76

🔑 ID Coverage:
   └─ With TMDB ID: 32,534 (100.0%)
   └─ With IMDB ID: 32,245 (99.1%)

✨ Enrichment Status:
   └─ TMDB Enriched: 32,534 (100.0%)
   └─ OMDB Enriched: 4,699 (14.4%)
   └─ Fully Refreshed: 4,699 (14.4%)

❄️  Frozen Movies: 0


## 2. Enrichment Status by Year

Shows how many movies per year have:
- Base IDs (TMDB/IMDB)
- Full TMDB enrichment
- OMDB enrichment

In [67]:
# Get enrichment breakdown by year
enrichment_df = get_enrichment_status_by_year()

# Display summary
print(f"\n📅 Enrichment Status for {len(enrichment_df)} Years\n")
print(enrichment_df.to_string(index=False))

# Show recent years in detail
print(f"\n\n🔍 Recent Years (2020+):\n")
recent = enrichment_df[enrichment_df['release_year'] >= 2020].copy()
if not recent.empty:
    print(recent.to_string(index=False))
else:
    print("No data for recent years yet.")


📅 Enrichment Status for 76 Years

 release_year  total_movies  with_base_ids  enriched_tmdb  enriched_omdb  pct_tmdb_enriched  pct_omdb_enriched
         2025           397            397            397            258              100.0              64.99
         2024           791            791            791            714              100.0              90.27
         2023           984            984            984            978              100.0              99.39
         2022          1083           1083           1083           1073              100.0              99.08
         2021          1092           1092           1092           1087              100.0              99.54
         2020          1038           1038           1038            589              100.0              56.74
         2019          1353           1353           1353              0              100.0               0.00
         2018          1390           1390           1390              0     

In [68]:
# Visualize enrichment status
if len(enrichment_df) > 0:
    # Filter to years with data (optional: adjust range as needed)
    plot_df = enrichment_df[enrichment_df['release_year'] >= 1980].copy()

    fig = go.Figure()

    # Add traces for each enrichment level
    fig.add_trace(go.Scatter(
        x=plot_df['release_year'],
        y=plot_df['total_movies'],
        name='Total Movies',
        mode='lines+markers',
        line=dict(width=2, color='lightgray'),
        marker=dict(size=6)
    ))

    fig.add_trace(go.Scatter(
        x=plot_df['release_year'],
        y=plot_df['enriched_tmdb'],
        name='TMDB Enriched',
        mode='lines+markers',
        line=dict(width=2, color='#01b4e4'),  # TMDB blue
        marker=dict(size=6)
    ))

    fig.add_trace(go.Scatter(
        x=plot_df['release_year'],
        y=plot_df['enriched_omdb'],
        name='OMDB Enriched',
        mode='lines+markers',
        line=dict(width=2, color='#f5c518'),  # IMDB yellow
        marker=dict(size=6)
    ))

    fig.update_layout(
        title='Data Enrichment Status by Year',
        xaxis_title='Release Year',
        yaxis_title='Number of Movies',
        hovermode='x unified',
        height=500
    )

    fig.show()
else:
    print("No data available for visualization.")

In [69]:
# Enrichment percentage heatmap (recent years)
if len(enrichment_df) > 0:
    recent_years = enrichment_df[enrichment_df['release_year'] >= 2000].copy()

    if not recent_years.empty:
        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=('TMDB Enrichment %', 'OMDB Enrichment %')
        )

        fig.add_trace(
            go.Bar(
                x=recent_years['release_year'],
                y=recent_years['pct_tmdb_enriched'],
                name='TMDB %',
                marker_color='#01b4e4'
            ),
            row=1, col=1
        )

        fig.add_trace(
            go.Bar(
                x=recent_years['release_year'],
                y=recent_years['pct_omdb_enriched'],
                name='OMDB %',
                marker_color='#f5c518'
            ),
            row=1, col=2
        )

        fig.update_yaxes(title_text='Percentage', range=[0, 105], row=1, col=1)
        fig.update_yaxes(title_text='Percentage', range=[0, 105], row=1, col=2)
        fig.update_xaxes(title_text='Year', row=1, col=1)
        fig.update_xaxes(title_text='Year', row=1, col=2)

        fig.update_layout(
            title_text='Enrichment Completion Rates (2000+)',
            showlegend=False,
            height=400
        )

        fig.show()

## 3. Movies Due for Update

Shows movies that need refreshing based on `movie_refresh_state` table.

In [70]:
# Get movies due for update by year
update_df = get_movies_due_for_update_by_year()

if len(update_df) > 0:
    total_due = update_df['movies_due_for_update'].sum()
    total_never = update_df['never_refreshed'].sum()
    total_overdue = update_df['overdue'].sum()

    print("="*60)
    print(" " * 15 + "UPDATE REQUIREMENTS")
    print("="*60)
    print(f"\n🔄 Total Movies Due for Update: {total_due:,}")
    print(f"   └─ Never Refreshed: {total_never:,}")
    print(f"   └─ Overdue: {total_overdue:,}")
    print("\n" + "="*60)

    print(f"\n📋 Breakdown by Year:\n")
    print(update_df.to_string(index=False))

    # Show top years needing updates
    top_10 = update_df.nlargest(10, 'movies_due_for_update')
    print(f"\n\n🔝 Top 10 Years Needing Updates:\n")
    print(top_10.to_string(index=False))
else:
    print("✅ All movies are up to date!")

               UPDATE REQUIREMENTS

🔄 Total Movies Due for Update: 32,534
   └─ Never Refreshed: 32,534
   └─ Overdue: 0


📋 Breakdown by Year:

 release_year  movies_due_for_update  never_refreshed  overdue
         2025                    397              397        0
         2024                    791              791        0
         2023                    984              984        0
         2022                   1083             1083        0
         2021                   1092             1092        0
         2020                   1038             1038        0
         2019                   1353             1353        0
         2018                   1390             1390        0
         2017                   1371             1371        0
         2016                   1285             1285        0
         2015                   1222             1222        0
         2014                   1179             1179        0
         2013                   1086

In [71]:
# Visualize update requirements
if len(update_df) > 0 and update_df['movies_due_for_update'].sum() > 0:
    # Filter to years with updates needed
    plot_df = update_df[update_df['movies_due_for_update'] > 0].copy()

    if len(plot_df) > 0:
        fig = go.Figure()

        fig.add_trace(go.Bar(
            x=plot_df['release_year'],
            y=plot_df['never_refreshed'],
            name='Never Refreshed',
            marker_color='#ff6b6b'
        ))

        fig.add_trace(go.Bar(
            x=plot_df['release_year'],
            y=plot_df['overdue'],
            name='Overdue',
            marker_color='#ffa726'
        ))

        fig.update_layout(
            title='Movies Requiring Updates by Year',
            xaxis_title='Release Year',
            yaxis_title='Number of Movies',
            barmode='stack',
            height=500,
            hovermode='x unified'
        )

        fig.show()
    else:
        print("No movies currently need updates.")
else:
    print("✅ No updates needed - database is current!")

## 4. Data Quality Metrics

Field completeness across different tables.

In [72]:
# Get data quality metrics
quality_df = get_data_quality_metrics()

print("\n📈 Data Quality Metrics (Field Completeness %)\n")
print(quality_df.to_string(index=False))

# Visualize quality metrics
if len(quality_df) > 0:
    # Melt dataframe for easier plotting
    quality_melted = quality_df.melt(
        id_vars=['source_table', 'total_records'],
        var_name='metric',
        value_name='completeness_pct'
    )

    fig = px.bar(
        quality_melted,
        x='metric',
        y='completeness_pct',
        color='source_table',
        barmode='group',
        title='Data Quality: Field Completeness by Table',
        labels={'completeness_pct': 'Completeness (%)', 'metric': 'Field'},
        height=500
    )

    fig.update_layout(yaxis_range=[0, 105])
    fig.add_hline(y=90, line_dash="dash", line_color="green", annotation_text="Target: 90%")
    fig.show()


📈 Data Quality Metrics (Field Completeness %)

source_table  total_records  title_completeness  release_date_completeness  tmdb_id_completeness  imdb_id_completeness
Movies Table          32534               100.0                     100.00                100.00                 99.11
 TMDB Movies          32534               100.0                      99.80                 37.33                 41.50
 OMDB Movies           4791               100.0                      94.49                 99.73                 99.39


## 5. Recent Update Activity

Track database update activity over the last 14 days.

In [73]:
# Get recent update activity
activity_df = get_recent_update_activity(days=14)

print("\n📊 Update Activity (Last 14 Days)\n")
print(activity_df.to_string(index=False))

# Summary stats
total_tmdb = activity_df['tmdb_updates'].sum()
total_omdb = activity_df['omdb_updates'].sum()
total_full = activity_df['full_refreshes'].sum()

print(f"\n📈 14-Day Summary:")
print(f"   └─ TMDB Updates: {total_tmdb:,}")
print(f"   └─ OMDB Updates: {total_omdb:,}")
print(f"   └─ Full Refreshes: {total_full:,}")


📊 Update Activity (Last 14 Days)

      date  tmdb_updates  omdb_updates  full_refreshes
2025-12-01             0           916             916
2025-11-30         28767             0               0
2025-11-29            18           896             896
2025-11-28             0           989             989
2025-11-27          3749           898             898
2025-11-26             0             0               0
2025-11-25             0           997             997
2025-11-24             0             0               0
2025-11-23             0             3               3
2025-11-22             0             0               0
2025-11-21             0             0               0
2025-11-20             0             0               0
2025-11-19             0             0               0
2025-11-18             0             0               0

📈 14-Day Summary:
   └─ TMDB Updates: 32,534
   └─ OMDB Updates: 4,699
   └─ Full Refreshes: 4,699


In [74]:
# Visualize update activity
if len(activity_df) > 0:
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=activity_df['date'],
        y=activity_df['tmdb_updates'],
        name='TMDB Updates',
        mode='lines+markers',
        line=dict(width=2, color='#01b4e4')
    ))

    fig.add_trace(go.Scatter(
        x=activity_df['date'],
        y=activity_df['omdb_updates'],
        name='OMDB Updates',
        mode='lines+markers',
        line=dict(width=2, color='#f5c518')
    ))

    fig.add_trace(go.Scatter(
        x=activity_df['date'],
        y=activity_df['full_refreshes'],
        name='Full Refreshes',
        mode='lines+markers',
        line=dict(width=2, color='#10b981')
    ))

    fig.update_layout(
        title='Database Update Activity (Last 14 Days)',
        xaxis_title='Date',
        yaxis_title='Number of Updates',
        hovermode='x unified',
        height=450
    )

    fig.show()


## 6. Custom Queries

Space for ad-hoc analysis and custom queries.

In [75]:
# Example: Top genres by count
query = """
SELECT
    genre,
    COUNT(*) as movie_count
FROM tmdb_movies,
    UNNEST(string_split(genres, ', ')) as t(genre)
WHERE genres IS NOT NULL AND genres != ''
GROUP BY genre
ORDER BY movie_count DESC
LIMIT 15
"""

genre_df = execute_custom_query(query)
print("\n🎬 Top 15 Genres by Movie Count:\n")
print(genre_df.to_string(index=False))

# Visualize
fig = px.bar(
    genre_df,
    x='movie_count',
    y='genre',
    orientation='h',
    title='Top Genres in Database',
    labels={'movie_count': 'Number of Movies', 'genre': 'Genre'}
)
fig.update_layout(height=500, yaxis={'categoryorder': 'total ascending'})
fig.show()



🎬 Top 15 Genres by Movie Count:

               genre  movie_count
              Comedy         2495
               Drama         2431
        Comedy,Drama          960
       Drama,Romance          954
         Documentary          786
              Horror          757
      Comedy,Romance          703
     Horror,Thriller          649
        Drama,Comedy          434
Comedy,Drama,Romance          423
      Drama,Thriller          389
       Drama,History          339
       Romance,Drama          275
      Romance,Comedy          271
     Action,Thriller          257


In [76]:
# Example: Movies added per month (last 12 months)
query = """
SELECT
    DATE_TRUNC('month', created_at) as month,
    COUNT(*) as movies_added
FROM movies
WHERE created_at >= CURRENT_DATE - INTERVAL '12 months'
GROUP BY month
ORDER BY month DESC
"""

monthly_df = execute_custom_query(query)
if len(monthly_df) > 0:
    print("\n📅 Movies Added per Month (Last 12 Months):\n")
    print(monthly_df.to_string(index=False))

    fig = px.line(
        monthly_df,
        x='month',
        y='movies_added',
        title='Database Growth: Movies Added per Month',
        markers=True
    )
    fig.update_layout(height=400)
    fig.show()
else:
    print("No recent movie additions found.")


📅 Movies Added per Month (Last 12 Months):

     month  movies_added
2025-11-01         32534


## Summary and Recommendations

Review the cells above to identify:
- **Years with low enrichment rates** - prioritize these for data collection
- **Movies due for updates** - consider running refresh workflows
- **Data quality gaps** - identify fields with low completeness
- **Recent activity trends** - ensure collection pipelines are running

---

*Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*